# Run `41ad591ec0de`

**Mode**: `research`
**Tier**: `standard`
**Datasets**: `tasic2018-v1`

## Request

> [bench:research-easy-tasic-celltype-counts] Count cells per cell type (use the 'cluster' field) in the Tasic 2018 V1 dataset. Produce: /work/celltype_counts.csv (two columns: cell_type, n_cells, sorted descending), and /work/celltype_counts_bar.png (horizontal bar plot of the top 30 types). Use the V1-only object pointed at by the dataset's access snippet.

## Final summary

All outputs verified. Here's the final report:

---

## Summary
Loaded the Tasic 2018 VISp-only processed object (`v1_all_proc.h5ad`, 13,586 cells) and counted cells across all 118 `cell_cluster` types. The dominant type is **L4 IT VISp Rspo1** (1,350 cells, ~10%), followed by **L6 IT VISp Penk Col27a1** (1,031) and **L2/3 IT VISp Agmat** (658). The long tail includes 5 types with only 1 cell each (mostly ALM contamination).

## What I did
- Loaded `/data/tasic2018-v1/v1_all_proc.h5ad` (13,586 VISp cells × 27,505 genes)
- Counted cells per `cell_cluster`, sorted descending — 118 unique types, all 13,586 cells accounted for
- Saved **`/work/celltype_counts.csv`** — 118 rows, two columns (`cell_type`, `n_cells`), monotonically sorted ✓
- Saved **`/work/celltype_counts_bar.png`** — horizontal bar chart of top 30 types (154 KB), colour-coded by broad class (blue = excitatory/projection, orange = inhibitory interneuron, green = non-neuronal)

## Caveats / next steps
- The V1 object retains a handful of cells labelled as ALM types (e.g. `L6 IT ALM Tgfb1`, n=1) — these appear to be edge-case assignments in the taxonomy, not a loading error; filter on `dissected_region == 'VISp'` if strict region purity is needed.
- Colour coding in the bar plot is heuristic (string-match based), not from a curated class field — a few ambiguous names may be mis-coloured; use `cell_class` obs column for authoritative class labels.
- The plot shows the top 30 of 118 types; the full table is in the CSV.

In [ ]:

import os
os.environ["NUMBA_DISABLE_JIT"] = "1"
import anndata as ad

adata = ad.read_h5ad("/data/tasic2018-v1/v1_all_proc.h5ad")
print(adata)
print("\nobs columns:", list(adata.obs.columns))
print("n_obs:", adata.n_obs)


exit_code: 0
--- stdout ---
AnnData object with n_obs × n_vars = 13586 × 27505
    obs: 'title', 'source_name', 'organism', 'donor_id', 'donor_sex', 'donor_genotype', 'injection_type', 'injection_target', 'injected_material', 'dissected_region', 'dissected_layer', 'facs_gating', 'facs_date', 'rna_amplification_set', 'sequencing_tube', 'sequencing_batch', 'sequencing_qc_pass_fail', 'cell_class', 'cell_subclass', 'cell_cluster', 'molecule', 'SRA_Run', 'GEO_Sample', 'GEO_Sample_Title', 'leiden'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'cell_class_colors', 'cell_subclass_colors', 'hvg', 'leiden', 'log1p', 'neighbors', 'normalization', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'cpm'
    obsp: 'connectivities', 'distances'

obs columns: ['title', 'source_name', 'organism', 'donor_id', 'donor_sex', 'donor_genotype', 'injection_type', 'injection_target', 'injected_material', 'dissected_r

In [ ]:

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- counts ---
counts = (
    adata.obs["cell_cluster"]
    .value_counts()
    .reset_index()
)
counts.columns = ["cell_type", "n_cells"]
counts = counts.sort_values("n_cells", ascending=False).reset_index(drop=True)

print(f"Total cell types: {len(counts)}")
print(f"Total cells:      {counts['n_cells'].sum()}")
print(counts.head(10).to_string())

# --- save CSV ---
counts.to_csv("/work/celltype_counts.csv", index=False)
print("\nSaved /work/celltype_counts.csv")


exit_code: 0
--- stdout ---
Total cell types: 118
Total cells:      13586
                    cell_type  n_cells
0            L4 IT VISp Rspo1     1350
1     L6 IT VISp Penk Col27a1     1031
2          L2/3 IT VISp Agmat      658
3                  Lamp5 Lsp1      425
4            L5 IT VISp Batf3      390
5                  Pvalb Tpbg      386
6  L6 IT VISp Col23a1 Adamts2      371
7                  Astro Aqp4      361
8             Pvalb Reln Tac1      328
9        L6 CT VISp Ctxn3 Sla      315

Saved /work/celltype_counts.csv

--- stderr ---
mkdir -p failed for path /home/sandbox/.config/matplotlib: [Errno 13] Permission denied: '/home/sandbox/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-f3d6pb0a because there was an issue with the default path (/home/sandbox/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better supp

In [ ]:

import os
os.environ["MPLCONFIGDIR"] = "/work/.mpl"
os.makedirs("/work/.mpl", exist_ok=True)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

top30 = counts.head(30).copy()

fig, ax = plt.subplots(figsize=(9, 10))
colors = ["#4C72B0" if "IT" in ct or "CT" in ct or "NP" in ct or "L" in ct[:2]
          else "#DD8452" if any(x in ct for x in ["Pvalb","Sst","Vip","Lamp5","Sncg","Meis2"])
          else "#55A868"  # non-neuronal
          for ct in top30["cell_type"]]

bars = ax.barh(top30["cell_type"][::-1], top30["n_cells"][::-1], color=colors[::-1], height=0.75)

# annotate counts
for bar, n in zip(bars, top30["n_cells"][::-1]):
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height()/2,
            str(n), va="center", ha="left", fontsize=8)

ax.set_xlabel("Number of cells", fontsize=11)
ax.set_title("Tasic 2018 — VISp cell type counts (top 30 of 118)", fontsize=13, fontweight="bold")
ax.set_xlim(0, top30["n_cells"].max() * 1.13)
ax.tick_params(axis="y", labelsize=8.5)
ax.spines[["top","right"]].set_visible(False)

# legend patches
import matplotlib.patches as mpatches
legend_handles = [
    mpatches.Patch(color="#4C72B0", label="Excitatory / projection"),
    mpatches.Patch(color="#DD8452", label="Inhibitory interneuron"),
    mpatches.Patch(color="#55A868", label="Non-neuronal"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig("/work/celltype_counts_bar.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved /work/celltype_counts_bar.png")


exit_code: 0
--- stdout ---
Saved /work/celltype_counts_bar.png

--- stderr ---



In [ ]:

import os

# verify CSV
csv_df = pd.read_csv("/work/celltype_counts.csv")
print("CSV shape:", csv_df.shape)
print("Sorted descending?", csv_df["n_cells"].is_monotonic_decreasing)
print("Sum of n_cells:", csv_df["n_cells"].sum())
print("\nBottom 5 types:")
print(csv_df.tail(5).to_string(index=False))

# verify PNG
png_size = os.path.getsize("/work/celltype_counts_bar.png")
print(f"\nPNG size: {png_size/1024:.1f} KB")


exit_code: 0
--- stdout ---
CSV shape: (118, 2)
Sorted descending? True
Sum of n_cells: 13586

Bottom 5 types:
           cell_type  n_cells
     L6 IT ALM Tgfb1        1
       L5 IT ALM Npw        1
L5 IT ALM Cpa6 Gpr88        1
     L2/3 IT ALM Sla        1
      L6 CT ALM Cpa6        1

PNG size: 154.8 KB

--- stderr ---



## Artifacts
- [`celltype_counts.csv`](./celltype_counts.csv)
- [`celltype_counts_bar.png`](./celltype_counts_bar.png)